# Basic Operations on Bayesian Networks

This notebook shows examples of some basic operations that can be performed on a Bayesian Network. We use the Protein Signalling network from the bnlearn repository as the example model: https://www.bnlearn.com/bnrepository/discrete-medium.html#sachs


The `DiscreteBayesianNetwork` class in pgmpy inherits the `networkx.DiGraph` class, hence all the methods defined for `networkx.DiGraph` should also work for `DiscreteBayesianNetwork`.

In [1]:
import pprint
import importlib
import networkx as nx
from pgmpy.factors.discrete import TabularCPD
from pgmpy.example_models import load_model
from pgmpy.examples_utils import VisualizationHelper, ModelInspector

# Load the sachs model
sachs_model = load_model('bnlearn/sachs')

# Visualize the model with graceful fallback
output_file = VisualizationHelper.visualize_graph(sachs_model, output_file='sachs.png', prog='dot')
if output_file:
    VisualizationHelper.display_image(output_file)
else:
    print("Note: Visualization requires pygraphviz. Model loaded successfully.")

c:\dev\pgmpy\pgmpy-eu-summer-of-code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-31 10:24:36,298 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/pgmpy/example_models/resolve/main/discrete/sachs.bif.gz "HTTP/1.1 302 Found"
2026-03-31 10:24:36,300 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-31 10:24:37,748 - pgmpy.examples_utils - WARNING - ⚠ pygraphviz not installed. Install with: pip install pygraphviz


Note: Visualization requires pygraphviz. Model loaded successfully.


## Attributes of the Model Structure

In [2]:
# Get all the nodes/random variables in the model
all_nodes = sachs_model.nodes()
print(f"Nodes: {all_nodes} \n")

# Get all the edges in the model.
all_edges = sachs_model.edges()
print(f"Edges: {all_edges} \n")

# Get all the CPDs.
all_cpds = sachs_model.get_cpds()

# Get parents of a specific node
akt_parents = sachs_model.get_parents('Akt')
print(f"Parents of Akt: {akt_parents} \n")

# Get children of a specific node
pka_children = sachs_model.get_children('PKA')
print(f"Children of PKA: {pka_children} \n")

# Get all the leaf nodes of the model
leaves = sachs_model.get_leaves()
print(f"Leaf nodes in the model: {leaves} \n")

# Get the root nodes of the model
roots = sachs_model.get_roots()
print(f"Root nodes in the model: {roots} \n")

Nodes: ['Akt', 'Erk', 'Jnk', 'Mek', 'P38', 'PIP2', 'PIP3', 'PKA', 'PKC', 'Plcg', 'Raf'] 

Edges: [('Erk', 'Akt'), ('Mek', 'Erk'), ('PIP3', 'PIP2'), ('PKA', 'Akt'), ('PKA', 'Erk'), ('PKA', 'Jnk'), ('PKA', 'Mek'), ('PKA', 'P38'), ('PKA', 'Raf'), ('PKC', 'Jnk'), ('PKC', 'Mek'), ('PKC', 'P38'), ('PKC', 'PKA'), ('PKC', 'Raf'), ('Plcg', 'PIP2'), ('Plcg', 'PIP3'), ('Raf', 'Mek')] 

Parents of Akt: ['Erk', 'PKA'] 

Children of PKA: ['Akt', 'Erk', 'Jnk', 'Mek', 'P38', 'Raf'] 

Leaf nodes in the model: ['Akt', 'Jnk', 'P38', 'PIP2'] 

Root nodes in the model: ['PKC', 'Plcg'] 



## Modifying the Model Structure

In [3]:
# Adding nodes to the model.
sachs_model.add_node('new_node')
sachs_model.add_nodes_from(['new_node1', 'new_node2'])

# Adding edges to the model.
sachs_model.add_edge('Akt', 'new_node')
sachs_model.add_edges_from([('Akt', 'new_node1'), ('Akt', 'new_node2')])

# Removing edges from the model.
sachs_model.remove_edge('Akt', 'new_node')
sachs_model.remove_edges_from([('Akt', 'new_node1'), ('Akt', 'new_node2')])

# Removing nodes from the model
sachs_model.remove_node('new_node')
sachs_model.remove_nodes_from(['new_node1', 'new_node2'])

In [4]:
# At any point, check_model can be called to check if the specified model is correct.
sachs_model.check_model()

True

## Modifying associated parameterization

In [5]:
# Getting an associated CPD
sachs_model.get_cpds('Akt')

# Adding new CPDs to the model
sachs_model.add_node('new_node')
new_cpd = TabularCPD('new_node', 2, [[0.2], [0.8]])
sachs_model.add_cpds(new_cpd)

# Removing the CPD and the node
sachs_model.remove_cpds('new_node')
sachs_model.remove_node('new_node')

sachs_model.check_model()

True

## D-Separation

In [6]:
# Check if two variables in the network are conditionally / unconditionally d-connected.
print(sachs_model.is_dconnected('PKC', 'Akt'))
print(sachs_model.is_dconnected('PKC', 'Akt', observed=['Mek']))
print(sachs_model.is_dconnected('PKC', 'Akt', observed=['Mek', 'PKA']))

True
True
False


In [7]:
# List all the variables that are d-connected to a given variable.
print(sachs_model.active_trail_nodes('PKA'))
print(sachs_model.active_trail_nodes(['PKA', 'Raf']))

print()

# List all d-connected variables when conditioned on some other variables
print(sachs_model.active_trail_nodes('PKA', observed=['Mek', 'PKC']))
print(sachs_model.active_trail_nodes(['PKA', 'Raf'], observed=['Mek', 'PKC']))

{'PKA': {'P38', 'PKA', 'Akt', 'Erk', 'Mek', 'PKC', 'Jnk', 'Raf'}}
{'PKA': {'P38', 'PKA', 'Akt', 'Erk', 'Mek', 'PKC', 'Jnk', 'Raf'}, 'Raf': {'P38', 'PKA', 'Akt', 'Erk', 'Mek', 'PKC', 'Jnk', 'Raf'}}

{'PKA': {'Akt', 'Erk', 'Jnk', 'P38', 'PKA', 'Raf'}}
{'PKA': {'Akt', 'Erk', 'Jnk', 'P38', 'PKA', 'Raf'}, 'Raf': {'Akt', 'Erk', 'Jnk', 'P38', 'PKA', 'Raf'}}


In [8]:
# Find the minimal d-separator of any two variables
print(sachs_model.minimal_dseparator('PKC', 'Akt'))

{'Erk', 'PKA'}


## Other Methods

In [9]:
# Get the Markov blanket of a variable
sachs_model.get_markov_blanket('Raf')

['PKC', 'PKA', 'Mek']

In [10]:
# List all local indpeendencies of a node
sachs_model.local_independencies('Raf')

(Raf ⟂ PIP3, Jnk, Plcg, P38, PIP2 | PKC, PKA)

In [11]:
# List all implied independencies in the network
sachs_model.get_independencies().independencies[:10]

[(Plcg ⟂ Raf),
 (PIP2 ⟂ Erk),
 (PIP3 ⟂ Mek),
 (Akt ⟂ Raf | Erk, PKA),
 (PIP2 ⟂ Jnk),
 (Akt ⟂ Jnk | PKC, PKA),
 (Erk ⟂ Jnk | PKA, Mek),
 (Erk ⟂ Raf | PKA, Mek),
 (PIP3 ⟂ P38),
 (PIP2 ⟂ PKA)]